In [1]:
import pandas as pd
import numpy as np
import os
import gc
import sqlite3
import time
import hashlib

from rapidfuzz import fuzz

print("Imports complete")

Imports complete


In [2]:
S1_FILE = r"train_source1_blocked.tsv"
S2_FILE = r"D:\All Data of Desktop with documents\Amazon_Blocking\train_source2_blocked.tsv"
S3_FILE = r"D:\All Data of Desktop with documents\Amazon_Blocking\train_source3_blocked.tsv"

LABELED_FILE = r"D:\All Data of Desktop with documents\Amazon_Blocking\labeled_pairs.tsv"

DB_FILE = r"D:\All Data of Desktop with documents\Amazon_Blocking\entity_lookup.db"

FEATURE_FILE = r"D:\All Data of Desktop with documents\Amazon_Blocking\training_features.tsv"

CHUNK_SIZE = 50_000

print("Paths ready")

Paths ready


In [3]:
if os.path.exists(DB_FILE):
    print("Existing database found:")
    print(DB_FILE)
else:
    print("New database will be created:")
    print(DB_FILE)

New database will be created:
D:\All Data of Desktop with documents\Amazon_Blocking\entity_lookup.db


In [4]:
conn = sqlite3.connect(DB_FILE)

conn.execute("""
CREATE TABLE IF NOT EXISTS entities (
    entity_id TEXT PRIMARY KEY,
    name_clean TEXT,
    addr_clean TEXT,
    country TEXT
)
""")

conn.commit()

print("SQLite table ready")

SQLite table ready


In [6]:
def add_file_to_db(file_path, conn, chunk_size=100_000):

    total = 0
    t0 = time.time()

    for chunk in pd.read_csv(
        file_path,
        sep="\t",
        usecols=[
            "entity_id",
            "name_clean",
            "addr_clean",
            "country"
        ],
        chunksize=chunk_size
    ):

        chunk = chunk.fillna("")

        rows = list(
            chunk[
                [
                    "entity_id",
                    "name_clean",
                    "addr_clean",
                    "country"
                ]
            ].itertuples(index=False, name=None)
        )

        conn.executemany(
            """
            INSERT OR REPLACE INTO entities
            (entity_id, name_clean, addr_clean, country)
            VALUES (?, ?, ?, ?)
            """,
            rows
        )

        conn.commit()

        total += len(chunk)

        print(
            f"Processed: {total:,}",
            f"| Time: {(time.time()-t0)/60:.2f} min"
        )

        del chunk
        del rows
        gc.collect()

    print("Completed:", file_path)

In [7]:
add_file_to_db(S1_FILE, conn)

Processed: 100,000 | Time: 0.01 min
Processed: 200,000 | Time: 0.04 min
Processed: 300,000 | Time: 0.07 min
Processed: 400,000 | Time: 0.10 min
Processed: 500,000 | Time: 0.13 min
Processed: 600,000 | Time: 0.21 min
Processed: 700,000 | Time: 0.29 min
Processed: 800,000 | Time: 0.33 min
Processed: 900,000 | Time: 0.39 min
Processed: 1,000,000 | Time: 0.45 min
Processed: 1,100,000 | Time: 0.50 min
Processed: 1,200,000 | Time: 0.60 min
Processed: 1,300,000 | Time: 0.70 min
Processed: 1,400,000 | Time: 0.75 min
Processed: 1,500,000 | Time: 0.80 min
Processed: 1,600,000 | Time: 0.93 min
Processed: 1,700,000 | Time: 1.06 min
Processed: 1,800,000 | Time: 1.13 min
Processed: 1,900,000 | Time: 1.18 min
Processed: 2,000,000 | Time: 1.24 min
Processed: 2,100,000 | Time: 1.35 min
Processed: 2,200,000 | Time: 1.47 min
Processed: 2,206,821 | Time: 1.48 min
Completed: train_source1_blocked.tsv


In [8]:
add_file_to_db(S2_FILE, conn)

Processed: 100,000 | Time: 0.01 min
Processed: 200,000 | Time: 0.04 min
Processed: 300,000 | Time: 0.07 min
Processed: 400,000 | Time: 0.10 min
Processed: 500,000 | Time: 0.13 min
Processed: 600,000 | Time: 0.16 min
Processed: 700,000 | Time: 0.19 min
Processed: 800,000 | Time: 0.23 min
Processed: 900,000 | Time: 0.26 min
Processed: 1,000,000 | Time: 0.30 min
Processed: 1,100,000 | Time: 0.33 min
Processed: 1,200,000 | Time: 0.37 min
Processed: 1,300,000 | Time: 0.41 min
Processed: 1,400,000 | Time: 0.45 min
Processed: 1,500,000 | Time: 0.49 min
Processed: 1,600,000 | Time: 0.54 min
Processed: 1,700,000 | Time: 0.58 min
Processed: 1,800,000 | Time: 0.62 min
Processed: 1,900,000 | Time: 0.67 min
Processed: 2,000,000 | Time: 0.71 min
Processed: 2,100,000 | Time: 0.76 min
Processed: 2,200,000 | Time: 0.81 min
Processed: 2,300,000 | Time: 0.85 min
Processed: 2,400,000 | Time: 0.90 min
Processed: 2,500,000 | Time: 0.95 min
Processed: 2,600,000 | Time: 1.00 min
Processed: 2,700,000 | Time: 1

In [9]:
add_file_to_db(S3_FILE, conn)

Processed: 100,000 | Time: 0.01 min
Processed: 200,000 | Time: 0.04 min
Processed: 300,000 | Time: 0.07 min
Processed: 400,000 | Time: 0.10 min
Processed: 500,000 | Time: 0.13 min
Processed: 600,000 | Time: 0.16 min
Processed: 700,000 | Time: 0.20 min
Processed: 800,000 | Time: 0.23 min
Processed: 900,000 | Time: 0.27 min
Processed: 1,000,000 | Time: 0.31 min
Processed: 1,100,000 | Time: 0.34 min
Processed: 1,200,000 | Time: 0.38 min
Processed: 1,300,000 | Time: 0.42 min
Processed: 1,400,000 | Time: 0.46 min
Processed: 1,500,000 | Time: 0.51 min
Processed: 1,600,000 | Time: 0.55 min
Processed: 1,700,000 | Time: 0.59 min
Processed: 1,800,000 | Time: 0.64 min
Processed: 1,900,000 | Time: 0.69 min
Processed: 2,000,000 | Time: 0.74 min
Processed: 2,100,000 | Time: 0.78 min
Processed: 2,200,000 | Time: 0.83 min
Processed: 2,300,000 | Time: 0.89 min
Processed: 2,400,000 | Time: 0.95 min
Processed: 2,500,000 | Time: 1.01 min
Processed: 2,600,000 | Time: 1.06 min
Processed: 2,700,000 | Time: 1

In [10]:
conn.execute("""
CREATE INDEX IF NOT EXISTS idx_entity_id
ON entities(entity_id)
""")

conn.commit()

count = conn.execute(
    "SELECT COUNT(*) FROM entities"
).fetchone()[0]

print("Total entities in database:", f"{count:,}")

Total entities in database: 12,527,040


In [11]:
conn.close()
gc.collect()

print("Database closed")

Database closed


In [12]:
def make_features(
    s1_name,
    s2_name,
    s1_addr,
    s2_addr,
    s1_country,
    s2_country
):

    s1_name = str(s1_name)
    s2_name = str(s2_name)

    s1_addr = str(s1_addr)
    s2_addr = str(s2_addr)

    s1_country = str(s1_country)
    s2_country = str(s2_country)

    name_ratio = fuzz.ratio(
        s1_name,
        s2_name
    )

    name_partial = fuzz.partial_ratio(
        s1_name,
        s2_name
    )

    name_token = fuzz.token_set_ratio(
        s1_name,
        s2_name
    )

    address_ratio = fuzz.ratio(
        s1_addr,
        s2_addr
    )

    address_partial = fuzz.partial_ratio(
        s1_addr,
        s2_addr
    )

    address_token = fuzz.token_set_ratio(
        s1_addr,
        s2_addr
    )

    return {
        "name_ratio": name_ratio,
        "name_partial_ratio": name_partial,
        "name_token_ratio": name_token,

        "name_length_diff": abs(
            len(s1_name) - len(s2_name)
        ),

        "address_ratio": address_ratio,
        "address_partial_ratio": address_partial,
        "address_token_ratio": address_token,

        "address_length_diff": abs(
            len(s1_addr) - len(s2_addr)
        ),

        "country_match": int(
            s1_country == s2_country
        ),

        "name_exact": int(
            s1_name == s2_name and s1_name != ""
        ),

        "address_exact": int(
            s1_addr == s2_addr and s1_addr != ""
        )
    }

In [13]:
FEATURE_COLUMNS = [
    "name_ratio",
    "name_partial_ratio",
    "name_token_ratio",
    "name_length_diff",

    "address_ratio",
    "address_partial_ratio",
    "address_token_ratio",
    "address_length_diff",

    "country_match",

    "name_exact",
    "address_exact"
]

print(FEATURE_COLUMNS)

['name_ratio', 'name_partial_ratio', 'name_token_ratio', 'name_length_diff', 'address_ratio', 'address_partial_ratio', 'address_token_ratio', 'address_length_diff', 'country_match', 'name_exact', 'address_exact']


In [14]:
if os.path.exists(FEATURE_FILE):
    os.remove(FEATURE_FILE)
    print("Old feature file deleted")
else:
    print("No old feature file found")

No old feature file found


In [15]:
t0 = time.time()

first_chunk = True
feature_chunk_no = 0
total_rows = 0

print("Feature engineering started...")

Feature engineering started...


In [17]:
def sqlite_lookup_batched(conn, ids, batch_size=500):
    """
    SQLite me IDs ko small batches me lookup karta hai
    taaki 'too many SQL variables' error na aaye.
    """

    lookup = {}

    for start in range(0, len(ids), batch_size):

        batch_ids = ids[start:start + batch_size]

        placeholders = ",".join(["?"] * len(batch_ids))

        query = f"""
            SELECT
                entity_id,
                name_clean,
                addr_clean,
                country
            FROM entities
            WHERE entity_id IN ({placeholders})
        """

        rows = conn.execute(
            query,
            batch_ids
        ).fetchall()

        for row in rows:
            lookup[str(row[0])] = {
                "name_clean": row[1] or "",
                "addr_clean": row[2] or "",
                "country": row[3] or ""
            }

    return lookup

In [21]:
import pandas as pd
import sqlite3
import gc
import time
import os

# =========================
# PATHS
# =========================

DB_FILE = r"D:\All Data of Desktop with documents\Amazon_Blocking\entity_lookup.db"
LABELED_FILE = r"D:\All Data of Desktop with documents\Amazon_Blocking\labeled_pairs.tsv"
FEATURE_FILE = r"D:\All Data of Desktop with documents\training_features.tsv"

CHUNK_SIZE = 50_000

# =========================
# CHECK FILES
# =========================

print("DB exists:", os.path.exists(DB_FILE))
print("Labeled file exists:", os.path.exists(LABELED_FILE))

# =========================
# RESET OUTPUT
# =========================

if os.path.exists(FEATURE_FILE):
    os.remove(FEATURE_FILE)
    print("Old feature file removed.")

# =========================
# SQLITE BATCH LOOKUP
# =========================

def sqlite_lookup_batched(conn, ids, batch_size=500):

    lookup = {}

    for start in range(0, len(ids), batch_size):

        batch_ids = ids[start:start + batch_size]

        placeholders = ",".join(
            ["?"] * len(batch_ids)
        )

        query = f"""
            SELECT
                entity_id,
                name_clean,
                addr_clean,
                country
            FROM entities
            WHERE entity_id IN ({placeholders})
        """

        rows = conn.execute(
            query,
            batch_ids
        ).fetchall()

        for row in rows:

            lookup[str(row[0])] = {
                "name_clean": row[1] or "",
                "addr_clean": row[2] or "",
                "country": row[3] or ""
            }

    return lookup


# =========================
# START
# =========================

conn = sqlite3.connect(DB_FILE)

first_chunk = True
feature_chunk_no = 0
total_rows = 0

t0 = time.time()

# =========================
# PROCESS LABELED FILE
# =========================

for pairs in pd.read_csv(
    LABELED_FILE,
    sep="\t",
    chunksize=CHUNK_SIZE
):

    feature_chunk_no += 1

    print(
        f"\nFeature Chunk {feature_chunk_no}"
        f" | Rows: {len(pairs):,}"
    )

    # -------------------------
    # COLLECT REQUIRED IDS
    # -------------------------

    ids = set(
        pairs["source1_entity_id"]
        .astype(str)
        .tolist()
    )

    ids.update(
        pairs["candidate_entity_id"]
        .astype(str)
        .tolist()
    )

    ids = list(ids)

    print(
        "Unique IDs:",
        f"{len(ids):,}"
    )

    # -------------------------
    # SQLITE BATCH LOOKUP
    # -------------------------

    lookup = sqlite_lookup_batched(
        conn,
        ids,
        batch_size=500
    )

    print(
        "IDs found:",
        f"{len(lookup):,}"
    )

    # -------------------------
    # FEATURE GENERATION
    # -------------------------

    output = []

    empty_entity = {
        "name_clean": "",
        "addr_clean": "",
        "country": ""
    }

    for row in pairs.itertuples(index=False):

        s1_id = str(row.source1_entity_id)
        cand_id = str(row.candidate_entity_id)

        s1 = lookup.get(
            s1_id,
            empty_entity
        )

        cand = lookup.get(
            cand_id,
            empty_entity
        )

        features = make_features(
            s1["name_clean"],
            cand["name_clean"],

            s1["addr_clean"],
            cand["addr_clean"],

            s1["country"],
            cand["country"]
        )

        features["source1_entity_id"] = s1_id
        features["candidate_entity_id"] = cand_id
        features["is_match"] = int(row.is_match)

        output.append(features)

    # -------------------------
    # DATAFRAME
    # -------------------------

    feature_df = pd.DataFrame(output)

    columns = [
        "source1_entity_id",
        "candidate_entity_id"
    ] + FEATURE_COLUMNS + [
        "is_match"
    ]

    feature_df = feature_df[columns]

    # -------------------------
    # SAVE
    # -------------------------

    feature_df.to_csv(
        FEATURE_FILE,
        sep="\t",
        index=False,
        mode="w" if first_chunk else "a",
        header=first_chunk
    )

    first_chunk = False

    total_rows += len(feature_df)

    # -------------------------
    # PROGRESS
    # -------------------------

    print(
        "Features generated:",
        f"{len(feature_df):,}"
    )

    print(
        "Total:",
        f"{total_rows:,}"
    )

    print(
        "Time:",
        f"{(time.time() - t0) / 60:.2f} min"
    )

    # -------------------------
    # MEMORY CLEANUP
    # -------------------------

    del pairs
    del ids
    del lookup
    del output
    del feature_df

    gc.collect()


# =========================
# CLOSE DATABASE
# =========================

conn.close()

print("\n================================")
print("FEATURE ENGINEERING COMPLETE")
print("================================")
print(
    "Total rows:",
    f"{total_rows:,}"
)
print(
    "Output:",
    FEATURE_FILE
)

DB exists: True
Labeled file exists: True

Feature Chunk 1 | Rows: 50,000
Unique IDs: 50,041
IDs found: 50,041
Features generated: 50,000
Total: 50,000
Time: 0.27 min

Feature Chunk 2 | Rows: 50,000
Unique IDs: 49,979
IDs found: 49,979
Features generated: 50,000
Total: 100,000
Time: 0.46 min

Feature Chunk 3 | Rows: 50,000
Unique IDs: 50,126
IDs found: 50,126
Features generated: 50,000
Total: 150,000
Time: 0.63 min

Feature Chunk 4 | Rows: 50,000
Unique IDs: 49,988
IDs found: 49,988
Features generated: 50,000
Total: 200,000
Time: 0.79 min

Feature Chunk 5 | Rows: 50,000
Unique IDs: 49,814
IDs found: 49,814
Features generated: 50,000
Total: 250,000
Time: 1.20 min

Feature Chunk 6 | Rows: 50,000
Unique IDs: 50,118
IDs found: 50,118
Features generated: 50,000
Total: 300,000
Time: 1.59 min

Feature Chunk 7 | Rows: 50,000
Unique IDs: 49,373
IDs found: 49,373
Features generated: 50,000
Total: 350,000
Time: 1.99 min

Feature Chunk 8 | Rows: 50,000
Unique IDs: 49,943
IDs found: 49,943
Features

In [22]:
print(
    "Feature file exists:",
    os.path.exists(FEATURE_FILE)
)

if os.path.exists(FEATURE_FILE):

    size_gb = (
        os.path.getsize(FEATURE_FILE)
        / (1024**3)
    )

    print(
        "Feature file size:",
        round(size_gb, 2),
        "GB"
    )

Feature file exists: True
Feature file size: 8.08 GB


In [23]:
sample = pd.read_csv(
    FEATURE_FILE,
    sep="\t",
    nrows=10
)

print(sample.to_string(index=False))

del sample
gc.collect()

source1_entity_id candidate_entity_id  name_ratio  name_partial_ratio  name_token_ratio  name_length_diff  address_ratio  address_partial_ratio  address_token_ratio  address_length_diff  country_match  name_exact  address_exact  is_match
     S1-925783039        S2-157377754  100.000000          100.000000        100.000000                 0      91.666667              95.652174           100.000000                    0              1           1              0         1
     S1-925783039        S3-997698194  100.000000          100.000000        100.000000                 0      76.543210              76.190476            85.714286                    9              1           1              0         1
     S1-925783039        S2-335180728   80.000000          100.000000        100.000000                 9      92.753623              89.230769            92.753623                    3              1           0              0         0
     S1-925783039         S2-28195834   90.90909

0

In [24]:
def get_split(entity_id):

    h = hashlib.md5(
        str(entity_id).encode()
    ).hexdigest()

    value = int(h[:8], 16) % 100

    if value < 80:
        return "train"

    return "valid"

In [25]:
import lightgbm as lgb

print("LightGBM version:", lgb.__version__)

LightGBM version: 4.7.0


In [26]:
TRAIN_FILE = r"D:\All Data of Desktop with documents\Amazon_Blocking\train_features.tsv"
VALID_FILE = r"D:\All Data of Desktop with documents\Amazon_Blocking\valid_features.tsv"

if os.path.exists(TRAIN_FILE):
    os.remove(TRAIN_FILE)

if os.path.exists(VALID_FILE):
    os.remove(VALID_FILE)

first_train = True
first_valid = True

train_rows = 0
valid_rows = 0

In [27]:
for chunk in pd.read_csv(
    FEATURE_FILE,
    sep="\t",
    chunksize=100_000
):

    chunk["split"] = chunk[
        "source1_entity_id"
    ].map(get_split)

    train_chunk = chunk[
        chunk["split"] == "train"
    ].drop(columns=["split"])

    valid_chunk = chunk[
        chunk["split"] == "valid"
    ].drop(columns=["split"])

    train_chunk.to_csv(
        TRAIN_FILE,
        sep="\t",
        index=False,
        mode="w" if first_train else "a",
        header=first_train
    )

    valid_chunk.to_csv(
        VALID_FILE,
        sep="\t",
        index=False,
        mode="w" if first_valid else "a",
        header=first_valid
    )

    first_train = False
    first_valid = False

    train_rows += len(train_chunk)
    valid_rows += len(valid_chunk)

    print(
        f"Train: {train_rows:,} | "
        f"Valid: {valid_rows:,}"
    )

    del chunk
    del train_chunk
    del valid_chunk

    gc.collect()

print("Split complete")

Train: 80,650 | Valid: 19,350
Train: 158,400 | Valid: 41,600
Train: 236,750 | Valid: 63,250
Train: 317,100 | Valid: 82,900
Train: 398,550 | Valid: 101,450
Train: 477,200 | Valid: 122,800
Train: 557,600 | Valid: 142,400
Train: 637,950 | Valid: 162,050
Train: 717,550 | Valid: 182,450
Train: 798,300 | Valid: 201,700
Train: 878,300 | Valid: 221,700
Train: 958,450 | Valid: 241,550
Train: 1,038,600 | Valid: 261,400
Train: 1,117,850 | Valid: 282,150
Train: 1,197,250 | Valid: 302,750
Train: 1,277,900 | Valid: 322,100
Train: 1,358,300 | Valid: 341,700
Train: 1,437,950 | Valid: 362,050
Train: 1,518,750 | Valid: 381,250
Train: 1,599,850 | Valid: 400,150
Train: 1,679,950 | Valid: 420,050
Train: 1,759,550 | Valid: 440,450
Train: 1,838,900 | Valid: 461,100
Train: 1,917,600 | Valid: 482,400
Train: 1,997,150 | Valid: 502,850
Train: 2,075,200 | Valid: 524,800
Train: 2,154,350 | Valid: 545,650
Train: 2,233,550 | Valid: 566,450
Train: 2,312,800 | Valid: 587,200
Train: 2,392,050 | Valid: 607,950
Train: 2,

In [28]:
train_df = pd.read_csv(
    TRAIN_FILE,
    sep="\t"
)

valid_df = pd.read_csv(
    VALID_FILE,
    sep="\t"
)

print("Train:", train_df.shape)
print("Valid:", valid_df.shape)

Train: (54435750, 14)
Valid: (13591450, 14)


In [29]:
X_COLUMNS = FEATURE_COLUMNS

X_train = train_df[X_COLUMNS]
y_train = train_df["is_match"]

X_valid = valid_df[X_COLUMNS]
y_valid = valid_df["is_match"]

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)

print("\nTrain labels:")
print(y_train.value_counts())

print("\nValidation labels:")
print(y_valid.value_counts())

X_train: (54435750, 11)
X_valid: (13591450, 11)

Train labels:
is_match
0    52396032
1     2039718
Name: count, dtype: int64

Validation labels:
is_match
0    13083120
1      508330
Name: count, dtype: int64


In [30]:
positive_count = int(y_train.sum())
negative_count = int((y_train == 0).sum())

scale_pos_weight = (
    negative_count / positive_count
    if positive_count > 0
    else 1
)

print("Positive:", positive_count)
print("Negative:", negative_count)
print("Scale pos weight:", scale_pos_weight)

Positive: 2039718
Negative: 52396032
Scale pos weight: 25.687880383464773


In [31]:
model = lgb.LGBMClassifier(
    objective="binary",

    n_estimators=1000,

    learning_rate=0.03,

    num_leaves=31,

    max_depth=-1,

    min_child_samples=50,

    subsample=0.8,

    colsample_bytree=0.8,

    reg_alpha=0.5,

    reg_lambda=1.0,

    scale_pos_weight=scale_pos_weight,

    random_state=42,

    n_jobs=-1
)

model.fit(
    X_train,
    y_train,

    eval_set=[
        (X_valid, y_valid)
    ],

    callbacks=[
        lgb.early_stopping(
            50
        ),
        lgb.log_evaluation(
            50
        )
    ]
)

print("Training complete")

d:\python\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 2039718, number of negative: 52396032
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.550162 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1667
[LightGBM] [Info] Number of data points in the train set: 54435750, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.037470 -> initscore=-3.246019
[LightGBM] [Info] Start training from score -3.246019
Training until validation scores don't improve for 50 rounds
[50]	valid_0's binary_logloss: 0.0538494
[100]	valid_0's binary_logloss: 0.0566602
Early stopping, best iteration is:
[57]	valid_0's binary_logloss: 0.0537339
Training complete


In [32]:
valid_prob = model.predict_proba(
    X_valid
)[:, 1]

print("Predictions generated")
print(valid_prob[:20])

Predictions generated
[0.83280151 0.83280151 0.82060379 0.00834414 0.00731164 0.00731164
 0.00731164 0.0073521  0.01388232 0.00731164 0.00737905 0.00731164
 0.00731164 0.00731164 0.00731164 0.02831216 0.00739446 0.00731164
 0.00731164 0.00731164]


In [33]:
from sklearn.metrics import precision_score, recall_score

def f05_score(y_true, y_pred):

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    if precision == 0 and recall == 0:
        return 0

    return (
        1.25 * precision * recall
    ) / (
        0.25 * precision + recall
    )

In [34]:
threshold_results = []

for threshold in np.arange(
    0.10,
    0.991,
    0.01
):

    pred = (
        valid_prob >= threshold
    ).astype(int)

    precision = precision_score(
        y_valid,
        pred,
        zero_division=0
    )

    recall = recall_score(
        y_valid,
        pred,
        zero_division=0
    )

    f05 = f05_score(
        y_valid,
        pred
    )

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f05": f05
    })

threshold_df = pd.DataFrame(
    threshold_results
)

threshold_df = threshold_df.sort_values(
    "f05",
    ascending=False
)

print(
    threshold_df.head(20)
)

    threshold  precision    recall       f05
72       0.82   0.924130  0.766105  0.887516
71       0.81   0.881358  0.847270  0.874322
70       0.80   0.860015  0.881486  0.864225
69       0.79   0.844607  0.896416  0.854484
68       0.78   0.837442  0.903809  0.849924
67       0.77   0.832128  0.909238  0.846486
66       0.76   0.825808  0.913822  0.842028
65       0.75   0.817933  0.918423  0.836232
73       0.83   0.957858  0.546045  0.832316
64       0.74   0.812314  0.921956  0.832105
63       0.73   0.802966  0.925493  0.824805
62       0.72   0.792925  0.931155  0.817187
61       0.71   0.741232  0.951832  0.775551
60       0.70   0.731552  0.955242  0.767497
59       0.69   0.720728  0.958440  0.758345
58       0.68   0.715455  0.960272  0.753896
57       0.67   0.692861  0.967769  0.734596
56       0.66   0.686120  0.969762  0.728750
55       0.65   0.681137  0.971509  0.724443
54       0.64   0.675925  0.973271  0.719913


In [35]:
best_row = threshold_df.iloc[0]

BEST_THRESHOLD = float(
    best_row["threshold"]
)

BEST_F05 = float(
    best_row["f05"]
)

BEST_PRECISION = float(
    best_row["precision"]
)

BEST_RECALL = float(
    best_row["recall"]
)

print("================================")
print("BEST VALIDATION RESULT")
print("================================")

print(
    "Threshold:",
    BEST_THRESHOLD
)

print(
    "Precision:",
    BEST_PRECISION
)

print(
    "Recall:",
    BEST_RECALL
)

print(
    "F0.5:",
    BEST_F05
)

BEST VALIDATION RESULT
Threshold: 0.8199999999999996
Precision: 0.9241301737516789
Recall: 0.7661046957684968
F0.5: 0.8875163289658762


In [36]:
importance = pd.DataFrame({
    "feature": X_COLUMNS,
    "importance": model.feature_importances_
})

importance = importance.sort_values(
    "importance",
    ascending=False
)

print(importance)

                  feature  importance
6     address_token_ratio         288
1      name_partial_ratio         262
3        name_length_diff         255
2        name_token_ratio         243
4           address_ratio         207
0              name_ratio         200
5   address_partial_ratio         158
7     address_length_diff          97
8           country_match           0
9              name_exact           0
10          address_exact           0


In [37]:
MODEL_PATH = r"D:\All Data of Desktop with documents\Amazon_Blocking\entity_match_lightgbm.txt"

model.booster_.save_model(
    MODEL_PATH
)

print("Model saved:")
print(MODEL_PATH)

Model saved:
D:\All Data of Desktop with documents\Amazon_Blocking\entity_match_lightgbm.txt


In [38]:
final_pred = (
    valid_prob >= BEST_THRESHOLD
).astype(int)

precision = precision_score(
    y_valid,
    final_pred,
    zero_division=0
)

recall = recall_score(
    y_valid,
    final_pred,
    zero_division=0
)

f05 = f05_score(
    y_valid,
    final_pred
)

print("================================")
print("FINAL VALIDATION")
print("================================")

print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F0.5      : {f05:.6f}")
print(f"Threshold : {BEST_THRESHOLD:.2f}")

FINAL VALIDATION
Precision : 0.924130
Recall    : 0.766105
F0.5      : 0.887516
Threshold : 0.82
